In [0]:
%pip install phonenumbers pycountry rapidfuzz

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import Window


In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_orders") # A replacer par silver clean par suppliers quand existant

# Lecture Silver
df_suppliers = spark.table("workspace.final_project.bronze_suppliers") # A remplacer par silver quand existant


In [0]:
### ORDER_ID & SUPPLIER_ID TO INT

def clean_integer_column(df, col_name, col_name_clean=None):
    """
    Clean a column containing integers possibly written as strings with .0 or .00.
    Valid values: 123, 123.0, 123.00
    Invalid values are replaced by null.

    Parameters:
        df : DataFrame
        col_name : str, column to clean
        col_name_clean : str or None, if provided create a new column with this name,
                         otherwise overwrite col_name
    """
    
    clean_str = F.regexp_replace(F.col(col_name).cast("string"), r"\.0+$", "")
    cleaned_col = F.when(
        F.col(col_name).cast("string").rlike(r"^[0-9]+(\.0+)?$"),
        clean_str.cast("int")
    ).otherwise(None)
    
    if col_name_clean:
        return df.withColumn(col_name_clean, cleaned_col)
    else:
        return df.withColumn(col_name, cleaned_col)

df_int = clean_integer_column(df, "order_id", "order_id_int") # On garde order_id original pour pouvoir lemettre dans la table quarantine
df_int = clean_integer_column(df_int, "supplier_id")
display(df_int)

In [0]:
### NULL ORDER_ID

df_valid = df_int.filter(F.col("order_id_int").isNotNull())

# Add remaining invalid rows (not in df_valid) to quarantine
df_remaining_quarantine = df_int.join(
    df_valid.select("order_id_int"), 
    on=["order_id_int"], 
    how="left_anti"  # keep only rows not in df_valid
)

# Add quarantine_reason and timestamp for these remaining invalid rows
df_remaining_quarantine = df_remaining_quarantine.withColumn(
    "quarantine_reason",
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .otherwise("invalid_order_id")  # other invalid cases
).withColumn(
    "quarantine_timestamp",
    F.current_timestamp()
)

df_remaining_quarantine = df_remaining_quarantine.drop("order_id_int") # On garde le nombre de colonnes de départ en supprimant la colonne transformée

df_valid = df_valid.drop("order_id") \
                   .withColumnRenamed("order_id_int", "order_id") # On supprime la colonne d'origine et on la remplace par la colonne transformée



In [0]:
### FULL DUPLICATES
# Identify and handle full duplicated lines
# Define a window partitioned by all columns to detect duplicates
window_all = Window.partitionBy(df_valid.columns).orderBy(F.lit(1))

# Add a row number within each duplicate group
df_with_rownum = df_valid.withColumn("row_num", F.row_number().over(window_all))

# Separate first occurrences (keep) and full duplicates (quarantine)
df_first_occurrences = df_with_rownum.filter(F.col("row_num") == 1).drop("row_num")
df_full_duplicates = df_with_rownum.filter(F.col("row_num") > 1).drop("row_num")

# Add quarantine_reason and timestamp for fully duplicated lines
df_full_duplicates = df_full_duplicates.withColumn("quarantine_reason", F.lit("full_duplicated_line")) \
                                       .withColumn("quarantine_timestamp", F.current_timestamp())

# Initialize df_orders_quarantine with full duplicates
df_orders_quarantine = df_full_duplicates

In [0]:
"""

# Define df_valid with cleaned order_id and supplier_id
# Keep only rows where order_id and supplier_id are not null and order_id is not an invalid string
df_valid = df_first_occurrences.filter(
    (F.col("order_id").isNotNull()) &         # real null check
    (F.trim(F.col("order_id")) != "") &       # remove empty strings
    (F.lower(F.col("order_id")) != "null") &  # remove string "null"
    (F.lower(F.col("order_id")) != "none") &  # remove string "none"
    (F.col("supplier_id").isNotNull())        # supplier_id must not be null
)
"""

In [0]:

### TABLE SPLIT

# Split of items and orders dataset
# Select columns for orders
df_orders = df_first_occurrences.select(
    "order_id",
    "supplier_id",
    "order_date",
    "delivery_date_expected",
    "delivery_date_actual"
)

# Select columns for items
df_items = df_first_occurrences.select(
    "order_id",
    F.posexplode("items").alias("item_index", "item")
)

# Flatten items
df_items = df_items.select(
    "order_id",
    "item_index",
    "item.product_category",
    "item.quantity",
    "item.unit_price"
)

# Add item_id
# We add a unique identifier for each item with hash 
df_items = df_items.withColumn(
    "item_id",
    F.sha2(F.concat_ws("||", F.col("order_id"), F.col("item_index")), 256)
)

In [0]:
# ---------- Date parsing ----------

def parse_date_col(col_name):
    c = F.trim(F.col(col_name).cast("string"))

    return F.coalesce(
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/d"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/M/dd"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy/MM/d"))),

        F.to_date(F.try_to_timestamp(c, F.lit("dd-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-M-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("d-MM-yyyy"))),
        F.to_date(F.try_to_timestamp(c, F.lit("dd-M-yyyy"))),

        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-dd'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-MM-d'T'HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(c, F.lit("yyyy-M-dd'T'HH:mm:ss")))
    )


# ---------- Clean date columns ----------

df_orders_clean = (
    df_orders
    .withColumn("order_date", parse_date_col("order_date"))
    .withColumn("delivery_date_expected", parse_date_col("delivery_date_expected"))
    .withColumn("delivery_date_actual", parse_date_col("delivery_date_actual"))
)

In [0]:
### VERIFICATION COHERENCE DATES

from pyspark.sql import functions as F

def validate_date_order(df, date_late_col, date_early_col, primary_key):
    """
    Filter DataFrame rows where date_late_col >= date_early_col (if neither is null).
    Rows failing this condition are moved to a quarantine DataFrame with reason and timestamp.

    Parameters:
        df : Spark DataFrame
        date_late_col : str, the column that should be later (delivery_date_expected)
        date_early_col : str, the column that should be earlier (order_date)

    Returns:
        df_valid : DataFrame with valid rows
        df_quarantine : DataFrame with invalid rows and quarantine columns
    """

    # Step 1: valid rows
    df_valid = df.filter(
        (F.col(date_late_col).isNull()) | 
        (F.col(date_early_col).isNull()) |
        (F.col(date_late_col) >= F.col(date_early_col))
    )

    # Step 2: rows failing the condition
    df_quarantine = df.join(
        df_valid.select(primary_key),  # assuming order_id is unique
        on=primary_key,
        how="left_anti"
    ).withColumn(
        "quarantine_reason",
        F.lit(f"{date_late_col}_before_{date_early_col}")
    ).withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )

    return df_valid, df_quarantine

# Verification delivery_date_expected > order_date
df_orders_time_coherence, df_quarantine_time= validate_date_order(df_orders_clean,
                                                              date_late_col="delivery_date_expected",
                                                              date_early_col="order_date",
                                                              primary_key="order_id")

# Verification delivery_date_actual > order_date
df_orders_time_coherence, df_quarantine_time_2 = validate_date_order(df_orders_coherence,
                                                              date_late_col="delivery_date_actual",
                                                              date_early_col="order_date",
                                                              primary_key="order_id")



In [0]:
### VERIFICATION COHERENCE SUPPLIER_ID

from pyspark.sql import functions as F

def validate_foreign_key(df_main, df_reference, fk_col, pk_col, primary_key_main):
    """
    Validate a foreign key column in a main DataFrame against a reference DataFrame.
    Rows with invalid foreign keys are moved to a quarantine DataFrame with reason and timestamp.

    Parameters:
        df_main : Spark DataFrame containing the main table
        df_reference : Spark DataFrame containing the reference table (primary keys)
        fk_col : str, column name in df_main that is the foreign key
        pk_col : str, column name in df_reference that is the primary key
        primary_key_main : str, column name of the primary key in df_main (unique per row)

    Returns:
        df_valid : rows in df_main with fk_col valid
        df_quarantine : rows in df_main with invalid fk_col, plus quarantine_reason and quarantine_timestamp
    """

    # Step 1: keep only rows with valid foreign keys
    df_valid = df_main.join(
        df_reference.select(pk_col).distinct().withColumnRenamed(pk_col, fk_col),
        on=fk_col,
        how="inner"
    )

    # Step 2: rows with invalid foreign keys
    df_quarantine = df_main.join(
        df_valid.select(primary_key_main),
        on=primary_key_main,
        how="left_anti"
    ).withColumn(
        "quarantine_reason",
        F.lit(f"{fk_col}_not_in_reference")
    ).withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )

    return df_valid, df_quarantine

df_orders_valid, df_orders_quarantine = validate_foreign_key(df_main=df_orders_coherence,
                                                             df_reference=df_suppliers,
                                                             fk_col="supplier_id",
                                                             pk_col="supplier_id",
                                                             primary_key_main="order_id"
                                                             )

In [0]:
###
display(df_items)


# Gerer les doublons
# Gerer les missing data (au moins retour statistique)
# Gerer le type de quantity (mettre en int)
# Ou gérer les valeurs aberrantes ?
